# Perth Property Price Prediction - Model Training

**Author:** Haz Li
**Date:** August, 2025

## Objective
The goal of this notebook is to develop a machine learning model that can accurately predict the price of a property in Perth based on its key features. The process involves:
1.  **Loading Data:** Connecting to our structured MySQL database to fetch clean, relational data.
2.  **Feature Engineering:** Preparing the data for modeling, primarily through one-hot encoding of categorical variables.
3.  **Model Training:** Training a `RandomForestRegressor` model, which is well-suited for tabular data and robust to outliers.
4.  **Evaluation:** Assessing the model's performance using the R-squared (R²) metric.
5.  **Serialization:** Saving the trained model and the required data columns for deployment in our Flask web application.

In [2]:
# --- 1. Import Necessary Libraries ---

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import joblib  # For saving our model and columns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Set pandas display options for better viewing
pd.set_option('display.max_columns', 50)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Data Loading

We will connect directly to our clean, structured MySQL database. This ensures we are using the "single source of truth" that we established during the ETL phase. The SQL query will join the fact and dimension tables to create a rich, flat dataset suitable for machine learning.

In [3]:
# --- 2. Database Connection and Data Loading (Secure Version) ---

import os
from dotenv import load_dotenv

# Load environment variables from the .env file in the project's root directory.
load_dotenv()
print("Attempting to load environment variables from .env file...")

# Read database credentials securely from environment variables.
DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = 'localhost'
DB_PORT = '3306'
DB_NAME = 'perth_property_db'

# A critical check to ensure the password was found.
if not DB_PASS:
    raise ValueError("DB_PASS environment variable not found or is empty. Please ensure your .env file is correctly set up in the project root.")
else:
    print("Database password loaded successfully from environment variable.")

# Create the SQLAlchemy engine using the loaded credentials.
db_connection_str = f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(db_connection_str)

# SQL query to join all tables and create a comprehensive dataset for modeling.
query = """
    SELECT
        p.price,
        p.land_size,
        p.parking_spaces,
        p.distance_to_cbd,
        s.suburb_name,
        l.bedrooms,
        l.bathrooms,
        ps.primary_school_icsea,
        ss.secondary_school_icsea
    FROM
        FACT_Properties p
    JOIN DIM_Suburbs s ON p.suburb_id = s.suburb_id
    JOIN DIM_Layouts l ON p.layout_id = l.layout_id
    LEFT JOIN DIM_Primary_Schools ps ON p.primary_school_id = ps.primary_school_id
    LEFT JOIN DIM_Secondary_Schools ss ON p.secondary_school_id = ss.secondary_school_id;
"""

print("\nLoading data from database...")
# Use a try-except block for robust data loading.
try:
    df = pd.read_sql(text(query), engine)
    print("Data loaded successfully!")
    print(f"Dataset shape: {df.shape}")
except Exception as e:
    print(f"Error loading data: {e}")

# Display the first few rows and info to verify.
display(df.head())
df.info()

Attempting to load environment variables from .env file...
Database password loaded successfully from environment variable.

Loading data from database...
Data loaded successfully!
Dataset shape: (42954, 9)


,price,land_size,parking_spaces,distance_to_cbd,suburb_name,bedrooms,bathrooms,primary_school_icsea,secondary_school_icsea
0,395000.0,411,1,13800,Alexander Heights,3,1,996,977
1,625000.0,695,2,14299,Alexander Heights,6,3,994,1030
2,275000.0,181,1,13940,Alexander Heights,3,1,994,1030
3,455000.0,547,2,14609,Alexander Heights,4,2,994,1030
4,450000.0,714,3,13816,Alexander Heights,4,2,994,1010


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42954 entries, 0 to 42953
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   price                   42954 non-null  float64
 1   land_size               42954 non-null  int64  
 2   parking_spaces          42954 non-null  int64  
 3   distance_to_cbd         42954 non-null  int64  
 4   suburb_name             42954 non-null  object 
 5   bedrooms                42954 non-null  int64  
 6   bathrooms               42954 non-null  int64  
 7   primary_school_icsea    42954 non-null  int64  
 8   secondary_school_icsea  42954 non-null  int64  
dtypes: float64(1), int64(7), object(1)
memory usage: 2.9+ MB
